# Processing results from different VLMs

In [17]:
#imports
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score



## Local VLMs for scenario prediction (from frames)

In [19]:
import re

# Modified to search subfolders with pattern results_promptX_framerate
def load_results(directory):
    results = pd.DataFrame()
    full_results_mode = pd.DataFrame()
    full_results_last = pd.DataFrame()
    
    # Look for subfolders matching pattern results_prompt*_*
    if not os.path.exists(directory):
        print(f"Directory {directory} does not exist!")
        return results, full_results_mode, full_results_last
    
    # Pattern to extract prompt and framerate from folder name
    # Example: results_promptA_30 or results_promptB_15
    folder_pattern = re.compile(r'results_prompt([AB])_(\d+)')
    
    # Iterate through items in directory
    for item in os.listdir(directory):
        item_path = os.path.join(directory, item)
        
        # Check if it's a directory and matches the pattern
        if os.path.isdir(item_path):
            match = folder_pattern.match(item)
            if match:
                prompt = match.group(1)  # A or B
                framerate = match.group(2)  # numeric value
                
                print(f"Processing folder: {item} (prompt={prompt}, framerate={framerate})")
                
                # Now look for CSV files in this subfolder
                for filename in os.listdir(item_path):
                    if filename.startswith('results_') and filename.endswith('.csv'):
                        filepath = os.path.join(item_path, filename)
                        df = pd.read_csv(filepath)
                        
                        # Extract model name from filename
                        model_name = filename[8:-4]  # Remove 'results_' and '.csv'
                        
                        # Add columns for model, prompt, and framerate
                        df['model'] = model_name
                        df['prompt'] = prompt
                        df['framerate'] = framerate
                        
                        # Reorder columns to have model, prompt, framerate first
                        cols = ['model', 'prompt', 'framerate'] + [col for col in df.columns if col not in ['model', 'prompt', 'framerate']]
                        df = df[cols]
                        
                        # Group by video_name and take the mode of the predictions
                        df_grouped_mode = df.groupby('video_name').agg(lambda x: x.mode()[0] if not x.mode().empty else np.nan).reset_index()
                        full_results_mode = pd.concat([full_results_mode, df_grouped_mode], ignore_index=True)
                        
                        # Get last frame prediction for each video
                        df_grouped_last = df.sort_values('frame').groupby('video_name').tail(1).reset_index(drop=True)
                        full_results_last = pd.concat([full_results_last, df_grouped_last], ignore_index=True)
                        
                        # Add to full results
                        results = pd.concat([results, df], ignore_index=True)
    
    return results, full_results_mode, full_results_last

In [20]:
path = '../results/'
results, full_results_mode, full_results_last = load_results(path)

#remove NaN values from the results
print("Removing NaN values from the results...")
print(results.shape)
#drop rows if 'outcome_prediction' contains word "Error"
results = results[~results['outcome_prediction'].str.contains("Error", na=False)]
print(results.shape)

print("\n" + "="*50)
print("MODE PREDICTIONS")
print("="*50)
print(full_results_mode.shape)
full_results_mode = full_results_mode[~full_results_mode['outcome_prediction'].str.contains("Error", na=False)]
print("Mode results after removing NaN values:")
print(full_results_mode.shape)

print("\n" + "="*50)
print("LAST FRAME PREDICTIONS")
print("="*50)
print(full_results_last.shape)
full_results_last = full_results_last[~full_results_last['outcome_prediction'].str.contains("Error", na=False)]
print("Last frame results after removing NaN values:")
print(full_results_last.shape)
print("types of predictions:", full_results_last['outcome_prediction'].unique())

#save the results to a csv file
results.to_csv('all_predictions.csv', index=False)
#save the mode results to a csv file
full_results_mode.to_csv('all_predictions_mode.csv', index=False)
#save the last frame results to a csv file
full_results_last.to_csv('all_predictions_last.csv', index=False)

#see how many unique videos are for each model, prompt, and framerate combination
print("\n" + "="*50)
print("Unique videos per model/prompt/framerate:")
print("="*50)
unique_videos_per_model = results.groupby(['model', 'prompt', 'framerate'])['video_name'].nunique().reset_index()
print(unique_videos_per_model)

Processing folder: results_promptB_3 (prompt=B, framerate=3)
Processing folder: results_promptA_3 (prompt=A, framerate=3)
Removing NaN values from the results...
(12092, 6)
(12092, 6)

MODE PREDICTIONS
(125, 6)
Mode results after removing NaN values:
(125, 6)

LAST FRAME PREDICTIONS
(125, 6)
Last frame results after removing NaN values:
(125, 6)
types of predictions: ['Well' 'Poorly' 'well']

Unique videos per model/prompt/framerate:
          model prompt framerate  video_name
0  deepseek_ocr      A         3           7
1  deepseek_ocr      B         3          30
2        gemma3      A         3          11
3        gemma3      B         3          30
4   llavallama3      A         3          17
5   llavallama3      B         3          30


In [12]:
#now, per model/prompt/framerate, print how many videos are "Poorly" and how many are "Well"
print("="*50)
print("MODE PREDICTIONS - Poorly vs Well counts")
print("="*50)

# Get unique combinations of model, prompt, and framerate
for (model, prompt, framerate) in full_results_mode.groupby(['model', 'prompt', 'framerate']).groups.keys():
    print(f"Model: {model}, Prompt: {prompt}, Framerate: {framerate}")
    model_results = full_results_mode[
        (full_results_mode['model'] == model) & 
        (full_results_mode['prompt'] == prompt) & 
        (full_results_mode['framerate'] == framerate)
    ]
    poorly_count = model_results['outcome_prediction'].str.contains('Poorly', case=False, na=False).sum()
    well_count = model_results['outcome_prediction'].str.contains('Well', case=False, na=False).sum()
    print(f"  Poorly: {poorly_count}")
    print(f"  Well: {well_count}")
    #print unique values
    print(model_results['outcome_prediction'].unique())

MODE PREDICTIONS - Poorly vs Well counts
Model: deepseek_ocr, Prompt: A, Framerate: 3
  Poorly: 0
  Well: 7
['Well']
Model: deepseek_ocr, Prompt: B, Framerate: 3
  Poorly: 0
  Well: 30
['Well']
Model: gemma3, Prompt: A, Framerate: 3
  Poorly: 11
  Well: 0
['Poorly']
Model: gemma3, Prompt: B, Framerate: 3
  Poorly: 30
  Well: 0
['Poorly']
Model: llavallama3, Prompt: A, Framerate: 3
  Poorly: 2
  Well: 15
['Well' 'Poorly']
Model: llavallama3, Prompt: B, Framerate: 3
  Poorly: 3
  Well: 27
['Well' 'Poorly']


In [13]:
#for outcome_prediction, replace poorly with 1 and well with 0
#if it contains "poorly", make new list with 1, if it contains "well", make new list with 0
full_results_mode['outcome_prediction_numeric'] = full_results_mode['outcome_prediction'].apply(
    lambda x: 1 if 'poorly' in str(x).lower() else (0 if 'well' in str(x).lower() else np.nan)
)
#print unique results
print(full_results_mode['outcome_prediction_numeric'].unique())


#save the full results with numeric outcome prediction to a csv file
full_results_mode.to_csv('all_predictions_mode.csv', index=False)

full_results_mode

[0 1]


,video_name,model,prompt,framerate,frame,outcome_prediction,outcome_prediction_numeric
0,11_final.mp4,llavallama3,B,3,0,Well,0
1,12_final.mp4,llavallama3,B,3,0,Well,0
2,14_final.mp4,llavallama3,B,3,0,Well,0
3,15_final.mp4,llavallama3,B,3,0,Well,0
4,19_final.mp4,llavallama3,B,3,0,Well,0
...,...,...,...,...,...,...,...
120,22_final.mp4,gemma3,A,3,0,Poorly,1
121,24_final.mp4,gemma3,A,3,0,Poorly,1
122,29_final.mp4,gemma3,A,3,0,Poorly,1
123,30_final.mp4,gemma3,A,3,0,Poorly,1


In [14]:
#now, get the groundtruth and see if they got it right
#open csv with columns Video,Question Mapping,Average Class of Human Predicion,True Outcome,

gt_df = pd.read_csv('../../../dataset_scenarios/analyze_predictions.csv')

#map each video to its true outcome
gt_df = gt_df[['Video', 'True Outcome']].rename(columns={'Video': 'video_name', 'True Outcome': 'true_outcome'})
#video are name without the .mp4 extension, so we need to add it to each one
gt_df['video_name'] = gt_df['video_name'].apply(lambda x: x + '.mp4' if not x.endswith('.mp4') else x)
#print type of true_outcome
gt_df['true_outcome'] = gt_df['true_outcome'].astype(int)
#dictionary to map video names to true outcomes
gt_dict = dict(zip(gt_df['video_name'], gt_df['true_outcome']))

#add the "true_outcome" column to the full_results_mode dataframe
full_results_mode['true_outcome'] = full_results_mode['video_name'].map(gt_dict)

#add the "true_outcome" column to the full_results_last dataframe
full_results_last['outcome_prediction_numeric'] = full_results_last['outcome_prediction'].apply(
    lambda x: 1 if 'poorly' in str(x).lower() else (0 if 'well' in str(x).lower() else np.nan)
)
#print unique results
print(full_results_last['outcome_prediction_numeric'].unique())
#check if there are nan
print(full_results_last['outcome_prediction_numeric'].isna().sum())


full_results_last['true_outcome'] = full_results_last['video_name'].map(gt_dict)

print("Mode results with ground truth:")
print(full_results_mode.head())
print("\nLast frame results with ground truth:")
print(full_results_last.head())

[0 1]
0
Mode results with ground truth:
     video_name        model prompt framerate  frame outcome_prediction  \
0  11_final.mp4  llavallama3      B         3      0               Well   
1  12_final.mp4  llavallama3      B         3      0               Well   
2  14_final.mp4  llavallama3      B         3      0               Well   
3  15_final.mp4  llavallama3      B         3      0               Well   
4  19_final.mp4  llavallama3      B         3      0               Well   

   outcome_prediction_numeric  true_outcome  
0                           0             1  
1                           0             1  
2                           0             1  
3                           0             1  
4                           0             0  

Last frame results with ground truth:
         model prompt framerate    video_name  frame outcome_prediction  \
0  llavallama3      B         3   6_final.mp4    183               Well   
1  llavallama3      B         3  22_final.mp

In [15]:
full_results_mode.to_csv("full_results_mode.csv")
full_results_last.to_csv("full_results_last.csv")

In [16]:
# ===================================================================
# MODE PREDICTIONS PERFORMANCE vs Ground Truth
# ===================================================================
print("="*50)
print("MODE PREDICTIONS PERFORMANCE vs Ground Truth")
print("="*50)

#models and metrics df - now includes prompt and framerate columns
prediction_performance_mode_df = pd.DataFrame(columns=['model', 'prompt', 'framerate', 'accuracy', 'precision', 'recall', 'f1_score', 'poorly_ratio'])

#now, go video by video and see if the model got it right
correct_predictions = []

# Iterate over unique combinations of model, prompt, and framerate
for (model, prompt, framerate) in full_results_mode.groupby(['model', 'prompt', 'framerate']).groups.keys():
    model_results = full_results_mode[
        (full_results_mode['model'] == model) & 
        (full_results_mode['prompt'] == prompt) & 
        (full_results_mode['framerate'] == framerate)
    ]
    
    y_pred = model_results['outcome_prediction_numeric']
    y_true = []
    for index, row in model_results.iterrows():
        video_name = row['video_name']
        true_outcome = gt_dict.get(video_name, np.nan)
        y_true.append(true_outcome)

    
    y_pred = np.array(y_pred)
    y_true = np.array(y_true)

    #get accuracy, precision, recall, f1 score
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # Calculate poorly ratio (ratio of poorly/0 predictions to all predictions)
    poorly_count = (y_pred == 1).sum()  # 1 represents "poorly" in our encoding
    total_count = len(y_pred)
    poorly_ratio = poorly_count / total_count if total_count > 0 else 0

    print(f"Model: {model}, Prompt: {prompt}, Framerate: {framerate}")
    print(f"  Accuracy: {accuracy:.2f}")
    print(f"  Precision: {precision:.2f}")
    print(f"  Recall: {recall:.2f}")
    print(f"  F1 Score: {f1:.2f}")
    print(f"  Poorly Ratio: {poorly_ratio:.2f}")

    df_pred = pd.DataFrame({
        'model': [model],
        'prompt': [prompt],
        'framerate': [framerate],
        'accuracy': [accuracy],
        'precision': [precision],
        'recall': [recall],
        'f1_score': [f1],
        'poorly_ratio': [poorly_ratio]
    })
    prediction_performance_mode_df = pd.concat([prediction_performance_mode_df, df_pred], ignore_index=True)

#save the prediction performance to a csv file
prediction_performance_mode_df.to_csv('prediction_performance_mode.csv', index=False)
#save full results with true outcome to a csv file
full_results_mode.to_csv('all_predictions_mode_with_true_outcome.csv', index=False)

prediction_performance_mode_df

MODE PREDICTIONS PERFORMANCE vs Ground Truth
Model: deepseek_ocr, Prompt: A, Framerate: 3
  Accuracy: 0.29
  Precision: 0.00
  Recall: 0.00
  F1 Score: 0.00
  Poorly Ratio: 0.00
Model: deepseek_ocr, Prompt: B, Framerate: 3
  Accuracy: 0.57
  Precision: 0.00
  Recall: 0.00
  F1 Score: 0.00
  Poorly Ratio: 0.00
Model: gemma3, Prompt: A, Framerate: 3
  Accuracy: 0.73
  Precision: 0.73
  Recall: 1.00
  F1 Score: 0.84
  Poorly Ratio: 1.00
Model: gemma3, Prompt: B, Framerate: 3
  Accuracy: 0.43
  Precision: 0.43
  Recall: 1.00
  F1 Score: 0.60
  Poorly Ratio: 1.00
Model: llavallama3, Prompt: A, Framerate: 3
  Accuracy: 0.53
  Precision: 1.00
  Recall: 0.20
  F1 Score: 0.33
  Poorly Ratio: 0.12
Model: llavallama3, Prompt: B, Framerate: 3
  Accuracy: 0.60
  Precision: 0.67
  Recall: 0.15
  F1 Score: 0.25
  Poorly Ratio: 0.10


/tmp/ipykernel_1509562/1846525140.py:61: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  prediction_performance_mode_df = pd.concat([prediction_performance_mode_df, df_pred], ignore_index=True)


,model,prompt,framerate,accuracy,precision,recall,f1_score,poorly_ratio
0,deepseek_ocr,A,3,0.285714,0.000000,0.000000,0.000000,0.000000
1,deepseek_ocr,B,3,0.566667,0.000000,0.000000,0.000000,0.000000
2,gemma3,A,3,0.727273,0.727273,1.000000,0.842105,1.000000
3,gemma3,B,3,0.433333,0.433333,1.000000,0.604651,1.000000
4,llavallama3,A,3,0.529412,1.000000,0.200000,0.333333,0.117647
5,llavallama3,B,3,0.600000,0.666667,0.153846,0.250000,0.100000


In [12]:
# ===================================================================
# LAST FRAME PREDICTIONS PERFORMANCE vs Ground Truth
# ===================================================================
print("\n" + "="*50)
print("LAST FRAME PREDICTIONS PERFORMANCE vs Ground Truth")
print("="*50)

#models and metrics df - now includes prompt and framerate columns
prediction_performance_last_df = pd.DataFrame(columns=['model', 'prompt', 'framerate', 'accuracy', 'precision', 'recall', 'f1_score', 'poorly_ratio'])

#now, go video by video and see if the model got it right
# Iterate over unique combinations of model, prompt, and framerate
for (model, prompt, framerate) in full_results_last.groupby(['model', 'prompt', 'framerate']).groups.keys():
    model_results = full_results_last[
        (full_results_last['model'] == model) & 
        (full_results_last['prompt'] == prompt) & 
        (full_results_last['framerate'] == framerate)
    ]
    
    y_pred = model_results['outcome_prediction_numeric']
    #if has NaN values, print it and print video name
    if y_pred.isna().any():
        print(f"Model {model} (Prompt {prompt}, Framerate {framerate}) has NaN values in predictions for videos:")
        print(model_results[model_results['outcome_prediction_numeric'].isna()]['video_name'].tolist())
    y_true = []
    for index, row in model_results.iterrows():
        video_name = row['video_name']
        true_outcome = gt_dict.get(video_name, np.nan)
        y_true.append(true_outcome)

    
    y_pred = np.array(y_pred)
    y_true = np.array(y_true)

    #get accuracy, precision, recall, f1 score
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    
    # Calculate poorly ratio (ratio of poorly/0 predictions to all predictions)
    poorly_count = (y_pred == 1).sum()  # 1 represents "poorly" in our encoding
    total_count = len(y_pred)
    poorly_ratio = poorly_count / total_count if total_count > 0 else 0

    print(f"Model: {model}, Prompt: {prompt}, Framerate: {framerate}")
    print(f"  Accuracy: {accuracy:.2f}")
    print(f"  Precision: {precision:.2f}")
    print(f"  Recall: {recall:.2f}")
    print(f"  F1 Score: {f1:.2f}")
    print(f"  Poorly Ratio: {poorly_ratio:.2f}")

    df_pred = pd.DataFrame({
        'model': [model],
        'prompt': [prompt],
        'framerate': [framerate],
        'accuracy': [accuracy],
        'precision': [precision],
        'recall': [recall],
        'f1_score': [f1],
        'poorly_ratio': [poorly_ratio]
    })
    prediction_performance_last_df = pd.concat([prediction_performance_last_df, df_pred], ignore_index=True)

#save the prediction performance to a csv file
prediction_performance_last_df.to_csv('prediction_performance_last.csv', index=False)
#save full results with true outcome to a csv file
full_results_last.to_csv('all_predictions_last_with_true_outcome.csv', index=False)

prediction_performance_last_df


LAST FRAME PREDICTIONS PERFORMANCE vs Ground Truth
Model: deepseek_ocr, Prompt: A, Framerate: 3
  Accuracy: 0.29
  Precision: 0.00
  Recall: 0.00
  F1 Score: 0.00
  Poorly Ratio: 0.00
Model: deepseek_ocr, Prompt: B, Framerate: 3
  Accuracy: 0.57
  Precision: 0.00
  Recall: 0.00
  F1 Score: 0.00
  Poorly Ratio: 0.00
Model: gemma3, Prompt: A, Framerate: 3
  Accuracy: 0.73
  Precision: 0.78
  Recall: 0.88
  F1 Score: 0.82
  Poorly Ratio: 0.82
Model: gemma3, Prompt: B, Framerate: 3
  Accuracy: 0.43
  Precision: 0.43
  Recall: 0.92
  F1 Score: 0.59
  Poorly Ratio: 0.93
Model: llavallama3, Prompt: A, Framerate: 3
  Accuracy: 0.41
  Precision: 0.50
  Recall: 0.10
  F1 Score: 0.17
  Poorly Ratio: 0.12
Model: llavallama3, Prompt: B, Framerate: 3
  Accuracy: 0.57
  Precision: 0.50
  Recall: 0.23
  F1 Score: 0.32
  Poorly Ratio: 0.20


/tmp/ipykernel_857508/3971668024.py:63: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  prediction_performance_last_df = pd.concat([prediction_performance_last_df, df_pred], ignore_index=True)


,model,prompt,framerate,accuracy,precision,recall,f1_score,poorly_ratio
0,deepseek_ocr,A,3,0.285714,0.000000,0.000000,0.000000,0.000000
1,deepseek_ocr,B,3,0.566667,0.000000,0.000000,0.000000,0.000000
2,gemma3,A,3,0.727273,0.777778,0.875000,0.823529,0.818182
3,gemma3,B,3,0.433333,0.428571,0.923077,0.585366,0.933333
4,llavallama3,A,3,0.411765,0.500000,0.100000,0.166667,0.117647
5,llavallama3,B,3,0.566667,0.500000,0.230769,0.315789,0.200000


In [72]:
# ===================================================================
# Load Human Predictions and Setup for Model-Human Comparison
# ===================================================================
print("\n" + "="*50)
print("Loading Human Predictions and Setting up Mappings")
print("="*50)

# Load human predictions data
human_df = pd.read_csv('../../../dataset_scenarios/badidea_ground_truth.csv')

# Filter to only include specific participant IDs
valid_participant_ids = [1048, 1251, 1483, 1676, 2103, 2313, 2698, 2946, 3157, 3203, 3339, 3882, 5009, 5099, 5124, 5233, 5310, 6488, 7136, 7782, 7797, 8184, 8436, 8758, 8786, 9055, 9385, 9777, 9941]
print(f"Filtering to only include {len(valid_participant_ids)} specific participants")
print(f"Original human dataset size: {len(human_df)}")
human_df = human_df[human_df['participant_id'].isin(valid_participant_ids)]
print(f"Filtered human dataset size: {len(human_df)}")

# Get unique participant IDs (now filtered)
participant_ids = human_df['participant_id'].unique()
print(f"Number of participants after filtering: {len(participant_ids)}")

# Define metrics calculation function
def calculate_metrics(y_true, y_pred_prob):
    from sklearn.metrics import roc_auc_score

    # Convert to binary if needed
    y_pred = (np.array(y_pred_prob) >= 0.5).astype(int)
    y_true = np.array(y_true).astype(int)

    # Handle cases where only one class is present
    try:
        auc = roc_auc_score(y_true, y_pred_prob)
    except:
        auc = np.nan

    # Calculate metrics
    metrics = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'auc': auc,
        'mse': np.mean((y_true - y_pred_prob) ** 2),
        'mae': np.mean(np.abs(y_true - y_pred_prob))
    }
    return metrics

# Calculate average human metrics for reference
print("\nCalculating human baseline metrics...")

# Use the correct mapping from question format to video format  
gt_df_full = pd.read_csv('../../../dataset_scenarios/analyze_predictions.csv')
question_to_outcome = dict(zip(gt_df_full['Question Mapping'], gt_df_full['True Outcome']))

# Calculate metrics for each individual human participant
human_metrics = {}
for participant_id in participant_ids:
    participant_data = human_df[human_df['participant_id'] == participant_id]
    
    true_outcomes = []
    participant_predictions = []
    
    for _, row in participant_data.iterrows():
        question_id = row['response_video']  # This is already in q_X format
        
        if question_id in question_to_outcome:
            true_outcomes.append(question_to_outcome[question_id])
            participant_predictions.append(row['class'])

    if len(true_outcomes) > 0:
        human_metrics[participant_id] = calculate_metrics(
            np.array(true_outcomes),
            np.array(participant_predictions)
        )

# Calculate average metrics across all participants
avg_human_metrics = {}
for metric in ['accuracy', 'precision', 'recall', 'f1', 'auc', 'mse', 'mae']:
    values = [m[metric] for m in human_metrics.values() if not np.isnan(m[metric])]
    avg_human_metrics[metric] = np.mean(values) if values else np.nan

print(f"Human baseline calculated from {len(human_metrics)} participants")

# Read the ground truth file again to get the proper mapping
gt_df_full = pd.read_csv('../../../dataset_scenarios/analyze_predictions.csv')

# Create mapping from question format (q_X) to video format (X_final.mp4)
question_to_video = dict(zip(gt_df_full['Question Mapping'], gt_df_full['Video']))
print("Question to Video mapping (sample):")
for i, (q, v) in enumerate(list(question_to_video.items())[:5]):
    print(f"   {q} -> {v}")

# Create the reverse mapping: video format to question format
video_to_question = {v + '.mp4': q for q, v in question_to_video.items()}
print("\nVideo to Question mapping (sample):")
for i, (v, q) in enumerate(list(video_to_question.items())[:5]):
    print(f"   {v} -> {q}")

# Test the mapping with actual model data
print("\nTesting the fixed mapping:")
sample_videos = list(full_results_mode['video_name'].unique())[:3]
print("   Sample model videos:", sample_videos)
print("   Corresponding question IDs:", [video_to_question.get(v, "NOT FOUND") for v in sample_videos])


Loading Human Predictions and Setting up Mappings
Filtering to only include 29 specific participants
Original human dataset size: 865
Filtered human dataset size: 865
Number of participants after filtering: 29

Calculating human baseline metrics...
Human baseline calculated from 29 participants
Question to Video mapping (sample):
   q_2 -> 6_final
   q_3 -> 7_final
   q_4 -> 9_final
   q_5 -> 11_final
   q_6 -> 12_final

Video to Question mapping (sample):
   6_final.mp4 -> q_2
   7_final.mp4 -> q_3
   9_final.mp4 -> q_4
   11_final.mp4 -> q_5
   12_final.mp4 -> q_6

Testing the fixed mapping:
   Sample model videos: ['11_final.mp4', '12_final.mp4', '14_final.mp4']
   Corresponding question IDs: ['q_5', 'q_6', 'q_7']


In [ ]:
# ===================================================================
# Local Models vs Individual Human Predictions - MODE PREDICTIONS
# ===================================================================
print("\n" + "="*50)
print("MODE PREDICTIONS vs Individual Human Predictions")
print("="*50)

# Debug: Check what we have
print("Debug - Available data:")
print(f"full_results_mode shape: {full_results_mode.shape}")
print(f"full_results_mode columns: {full_results_mode.columns.tolist()}")

# Check for NaN values in predictions
print(f"NaN values in outcome_prediction_numeric: {full_results_mode['outcome_prediction_numeric'].isna().sum()}")
print(f"Non-NaN values in outcome_prediction_numeric: {full_results_mode['outcome_prediction_numeric'].notna().sum()}")

# Use the video_to_question mapping from the previous cell
# Create a mapping from video name to model predictions (MODE)
# Now organized by model, prompt, and framerate
video_to_model_preds_mode = {}

for (model, prompt, framerate) in full_results_mode.groupby(['model', 'prompt', 'framerate']).groups.keys():
    key = (model, prompt, framerate)
    model_data = full_results_mode[
        (full_results_mode['model'] == model) & 
        (full_results_mode['prompt'] == prompt) & 
        (full_results_mode['framerate'] == framerate)
    ]
    print(f"\nModel {model}, Prompt {prompt}, Framerate {framerate} - before filtering:")
    print(f"  Total rows: {len(model_data)}")
    print(f"  Rows with outcome_prediction_numeric: {model_data['outcome_prediction_numeric'].notna().sum()}")
    
    # Remove NaN values from model predictions
    model_data_clean = model_data.dropna(subset=['outcome_prediction_numeric'])
    print(f"  After dropping NaN: {len(model_data_clean)}")
    
    # Check for actual values
    if len(model_data_clean) > 0:
        print(f"  Sample predictions: {model_data_clean['outcome_prediction_numeric'].head().tolist()}")
        print(f"  Unique predictions: {model_data_clean['outcome_prediction_numeric'].unique()}")
    
    video_to_model_preds_mode[key] = dict(zip(model_data_clean['video_name'], model_data_clean['outcome_prediction_numeric']))
    print(f"  Valid predictions in mapping: {len(video_to_model_preds_mode[key])}")

# Calculate metrics for each model/prompt/framerate compared to each participant
model_vs_human_metrics_mode = {}

for key in video_to_model_preds_mode.keys():
    model, prompt, framerate = key
    model_vs_human_metrics_mode[key] = {}
    print(f"\nProcessing model: {model}, Prompt: {prompt}, Framerate: {framerate}")
    
    model_pred_count = 0
    total_matches = 0
    
    for participant_id in participant_ids:
        # Get this participant's predictions
        participant_data = human_df[human_df['participant_id'] == participant_id]
        
        # Create lists to store participant predictions and model predictions
        human_preds = []
        model_preds = []
        
        # Match participant responses with model predictions using the mapping
        for _, row in participant_data.iterrows():
            question_id = row['response_video']  # e.g., 'q_2'
            
            # Convert question ID to video name using the mapping
            if question_id in question_to_video:
                video_name = question_to_video[question_id] + '.mp4'  # Convert to video format
                
                # Check if this video has a model prediction
                if video_name in video_to_model_preds_mode[key]:
                    model_pred = video_to_model_preds_mode[key][video_name]
                    if not np.isnan(model_pred):
                        human_preds.append(row['class'])
                        model_preds.append(model_pred)
                        total_matches += 1
        
        model_pred_count += len(human_preds)
        
        # Calculate metrics if this participant has matching predictions
        if len(human_preds) > 0:
            try:
                model_vs_human_metrics_mode[key][participant_id] = calculate_metrics(
                    np.array(human_preds),
                    np.array(model_preds)
                )
            except Exception as e:
                print(f"   Error calculating metrics for participant {participant_id}: {e}")
    
    print(f"   Total matches: {total_matches}")

# Calculate average metrics for each model/prompt/framerate vs humans
avg_model_vs_human_mode = {}
std_model_vs_human_mode = {}
for key, participant_metrics in model_vs_human_metrics_mode.items():
    if participant_metrics:  # Only if there are metrics for this combination
        avg_model_vs_human_mode[key] = {}
        std_model_vs_human_mode[key] = {}
        for metric in ['accuracy', 'precision', 'recall', 'f1', 'auc', 'mse', 'mae']:
            values = [m[metric] for m in participant_metrics.values() if not np.isnan(m[metric])]
            avg_model_vs_human_mode[key][metric] = np.mean(values) if values else np.nan
            std_model_vs_human_mode[key][metric] = np.std(values) if values else np.nan

# Convert to DataFrame for easier comparison
if avg_model_vs_human_mode:
    # Create a multi-index dataframe
    avg_model_vs_human_mode_df = pd.DataFrame(avg_model_vs_human_mode).T
    avg_model_vs_human_mode_df.index = pd.MultiIndex.from_tuples(avg_model_vs_human_mode_df.index, names=['model', 'prompt', 'framerate'])
    avg_model_vs_human_mode_df = avg_model_vs_human_mode_df.reset_index()
    
    print("\n" + "="*50)
    print("RESULTS: MODE - Average Agreement with Individual Humans")
    print("="*50)
    print(avg_model_vs_human_mode_df.round(3))
    
    std_model_vs_human_mode_df = pd.DataFrame(std_model_vs_human_mode).T
    std_model_vs_human_mode_df.index = pd.MultiIndex.from_tuples(std_model_vs_human_mode_df.index, names=['model', 'prompt', 'framerate'])
    std_model_vs_human_mode_df = std_model_vs_human_mode_df.reset_index()
    
    print("\nStandard Deviation of Agreement with Individual Humans:")
    print(std_model_vs_human_mode_df.round(3))
    
    # Save results
    avg_model_vs_human_mode_df.to_csv('model_vs_human_agreement_mode.csv', index=False)
    print("\nResults saved to 'model_vs_human_agreement_mode.csv'")
else:
    print("\n❌ No valid metrics calculated - check data alignment issues")

In [ ]:
# ===================================================================
# Local Models vs Individual Human Predictions - LAST FRAME PREDICTIONS
# ===================================================================
print("\n" + "="*50)
print("LAST FRAME PREDICTIONS vs Individual Human Predictions")
print("="*50)

# Use the video_to_question mapping from the previous cell
# Create a mapping from video name to model predictions (LAST FRAME)
video_to_model_preds_last = {}
for (model, prompt, framerate) in full_results_last.groupby(['model', 'prompt', 'framerate']).groups.keys():
    key = (model, prompt, framerate)
    model_data = full_results_last[
        (full_results_last['model'] == model) & 
        (full_results_last['prompt'] == prompt) & 
        (full_results_last['framerate'] == framerate)
    ]
    # Remove NaN values from model predictions
    model_data_clean = model_data.dropna(subset=['outcome_prediction_numeric'])
    video_to_model_preds_last[key] = dict(zip(model_data_clean['video_name'], model_data_clean['outcome_prediction_numeric']))
    print(f"Model {model}, Prompt {prompt}, Framerate {framerate}: {len(video_to_model_preds_last[key])} valid predictions")

# Calculate metrics for each model/prompt/framerate compared to each participant
model_vs_human_metrics_last = {}

for key in video_to_model_preds_last.keys():
    model, prompt, framerate = key
    model_vs_human_metrics_last[key] = {}
    print(f"\nProcessing model: {model}, Prompt: {prompt}, Framerate: {framerate}")
    
    for participant_id in participant_ids:
        # Get this participant's predictions
        participant_data = human_df[human_df['participant_id'] == participant_id]
        
        # Create lists to store participant predictions and model predictions
        human_preds = []
        model_preds = []
        
        # Match participant responses with model predictions using the mapping
        for _, row in participant_data.iterrows():
            question_id = row['response_video']  # e.g., 'q_2'
            
            # Convert question ID to video name using the mapping
            if question_id in question_to_video:
                video_name = question_to_video[question_id] + '.mp4'  # Convert to video format
                
                # Check if this video has a model prediction
                if video_name in video_to_model_preds_last[key]:
                    model_pred = video_to_model_preds_last[key][video_name]
                    if not np.isnan(model_pred):
                        human_preds.append(row['class'])
                        model_preds.append(model_pred)
        
        # Calculate metrics if this participant has matching predictions
        if len(human_preds) > 0:
            try:
                model_vs_human_metrics_last[key][participant_id] = calculate_metrics(
                    np.array(human_preds),
                    np.array(model_preds)
                )
            except Exception as e:
                print(f"   Error calculating metrics for participant {participant_id}: {e}")

# Calculate average metrics for each model/prompt/framerate vs humans
avg_model_vs_human_last = {}
std_model_vs_human_last = {}
for key, participant_metrics in model_vs_human_metrics_last.items():
    if participant_metrics:  # Only if there are metrics for this combination
        avg_model_vs_human_last[key] = {}
        std_model_vs_human_last[key] = {}
        for metric in ['accuracy', 'precision', 'recall', 'f1', 'auc', 'mse', 'mae']:
            values = [m[metric] for m in participant_metrics.values() if not np.isnan(m[metric])]
            avg_model_vs_human_last[key][metric] = np.mean(values) if values else np.nan
            std_model_vs_human_last[key][metric] = np.std(values) if values else np.nan

# Convert to DataFrame for easier comparison
if avg_model_vs_human_last:
    avg_model_vs_human_last_df = pd.DataFrame(avg_model_vs_human_last).T
    avg_model_vs_human_last_df.index = pd.MultiIndex.from_tuples(avg_model_vs_human_last_df.index, names=['model', 'prompt', 'framerate'])
    avg_model_vs_human_last_df = avg_model_vs_human_last_df.reset_index()
    
    print("\n" + "="*50)
    print("RESULTS: LAST FRAME - Average Agreement with Individual Humans")
    print("="*50)
    print(avg_model_vs_human_last_df.round(3))
    
    std_model_vs_human_last_df = pd.DataFrame(std_model_vs_human_last).T
    std_model_vs_human_last_df.index = pd.MultiIndex.from_tuples(std_model_vs_human_last_df.index, names=['model', 'prompt', 'framerate'])
    std_model_vs_human_last_df = std_model_vs_human_last_df.reset_index()
    
    print("\nStandard Deviation of Agreement with Individual Humans:")
    print(std_model_vs_human_last_df.round(3))
    
    # Save results
    avg_model_vs_human_last_df.to_csv('model_vs_human_agreement_last.csv', index=False)
    print("\nResults saved to 'model_vs_human_agreement_last.csv'")
else:
    print("\n❌ No valid metrics calculated - check data alignment issues")

# ===================================================================
# SUMMARY: MODE vs LAST FRAME Comparison
# ===================================================================
print("\n" + "="*80)
print("SUMMARY: Comparison of MODE vs LAST FRAME Predictions")
print("="*80)

print("\n" + "="*80)
print("Performance vs Ground Truth")
print("="*80)

print("\nMODE Predictions:")
print(prediction_performance_mode_df[['model', 'prompt', 'framerate', 'accuracy', 'precision', 'recall', 'f1_score']].round(3))

print("\nLAST FRAME Predictions:")
print(prediction_performance_last_df[['model', 'prompt', 'framerate', 'accuracy', 'precision', 'recall', 'f1_score']].round(3))

print("\n" + "="*80)
print("Agreement with Individual Humans")
print("="*80)

print("\nMODE Predictions:")
if 'avg_model_vs_human_mode_df' in locals():
    print(avg_model_vs_human_mode_df[['model', 'prompt', 'framerate', 'accuracy', 'precision', 'recall', 'f1']].round(3))
else:
    print("MODE human agreement data not available")

print("\nLAST FRAME Predictions:")
if 'avg_model_vs_human_last_df' in locals():
    print(avg_model_vs_human_last_df[['model', 'prompt', 'framerate', 'accuracy', 'precision', 'recall', 'f1']].round(3))
else:
    print("LAST FRAME human agreement data not available")

print("\n" + "="*80)
print("Human Baseline (for reference)")
print("="*80)
if 'avg_human_metrics' in locals():
    human_df_metrics = pd.DataFrame(avg_human_metrics, index=['Average Individual Human'])
    print(human_df_metrics[['accuracy', 'precision', 'recall', 'f1']].round(3))
else:
    print("Human baseline data not available")

In [ ]:
# ===================================================================
# Detailed Comparison Table: MODE vs LAST FRAME
# ===================================================================
print("\n" + "="*80)
print("Detailed Comparison: MODE vs LAST FRAME (vs Ground Truth)")
print("="*80)

# Create comparison dataframe with prompt and framerate
comparison_data = []

# Get unique combinations
for (model, prompt, framerate) in prediction_performance_mode_df.groupby(['model', 'prompt', 'framerate']).groups.keys():
    mode_row = prediction_performance_mode_df[
        (prediction_performance_mode_df['model'] == model) & 
        (prediction_performance_mode_df['prompt'] == prompt) & 
        (prediction_performance_mode_df['framerate'] == framerate)
    ].iloc[0]
    
    last_row = prediction_performance_last_df[
        (prediction_performance_last_df['model'] == model) & 
        (prediction_performance_last_df['prompt'] == prompt) & 
        (prediction_performance_last_df['framerate'] == framerate)
    ].iloc[0]
    
    comparison_data.append({
        'Model': model,
        'Prompt': prompt,
        'Framerate': framerate,
        'Mode_Accuracy': mode_row['accuracy'],
        'Last_Accuracy': last_row['accuracy'],
        'Diff_Accuracy': last_row['accuracy'] - mode_row['accuracy'],
        'Mode_F1': mode_row['f1_score'],
        'Last_F1': last_row['f1_score'],
        'Diff_F1': last_row['f1_score'] - mode_row['f1_score']
    })

comparison_df = pd.DataFrame(comparison_data)
print(comparison_df.round(3))

# Save the comparison
comparison_df.to_csv('mode_vs_last_comparison.csv', index=False)
print("\nComparison table saved to 'mode_vs_last_comparison.csv'")

# Identify which method is better for each model/prompt/framerate
print("\n" + "="*80)
print("Which aggregation method performs better? (vs Ground Truth)")
print("="*80)
for _, row in comparison_df.iterrows():
    model = row['Model']
    prompt = row['Prompt']
    framerate = row['Framerate']
    if row['Diff_Accuracy'] > 0.01:
        print(f"{model:20s} (Prompt {prompt}, FR {framerate}): LAST FRAME better (Δ Accuracy: +{row['Diff_Accuracy']:.3f})")
    elif row['Diff_Accuracy'] < -0.01:
        print(f"{model:20s} (Prompt {prompt}, FR {framerate}): MODE better (Δ Accuracy: {row['Diff_Accuracy']:.3f})")
    else:
        print(f"{model:20s} (Prompt {prompt}, FR {framerate}): Similar performance (Δ Accuracy: {row['Diff_Accuracy']:.3f})")

# Overall statistics
print("\n" + "="*80)
print("Overall Statistics")
print("="*80)
print(f"Average Mode Accuracy: {comparison_df['Mode_Accuracy'].mean():.3f}")
print(f"Average Last Frame Accuracy: {comparison_df['Last_Accuracy'].mean():.3f}")
print(f"Average Difference: {comparison_df['Diff_Accuracy'].mean():.3f}")
print(f"\nCombinations where LAST FRAME is better: {(comparison_df['Diff_Accuracy'] > 0.01).sum()}/{len(comparison_df)}")
print(f"Combinations where MODE is better: {(comparison_df['Diff_Accuracy'] < -0.01).sum()}/{len(comparison_df)}")
print(f"Combinations with similar performance: {((comparison_df['Diff_Accuracy'] >= -0.01) & (comparison_df['Diff_Accuracy'] <= 0.01)).sum()}/{len(comparison_df)}")

In [ ]:
# ===================================================================
# Visualization: MODE vs LAST FRAME Performance Comparison
# Note: With prompt and framerate dimensions, we'll create faceted plots
# ===================================================================

# Create a combined label for easier visualization
prediction_performance_mode_df['label'] = prediction_performance_mode_df['model'] + '_P' + prediction_performance_mode_df['prompt'] + '_FR' + prediction_performance_mode_df['framerate'].astype(str)
prediction_performance_last_df['label'] = prediction_performance_last_df['model'] + '_P' + prediction_performance_last_df['prompt'] + '_FR' + prediction_performance_last_df['framerate'].astype(str)

fig, axes = plt.subplots(2, 2, figsize=(20, 12))
fig.suptitle('Local VLM Performance: MODE vs LAST FRAME Predictions (by Prompt & Framerate)', fontsize=16, fontweight='bold')

metrics_to_plot = ['accuracy', 'precision', 'recall', 'f1_score']
metric_titles = ['Accuracy', 'Precision', 'Recall', 'F1 Score']

for idx, (metric, title) in enumerate(zip(metrics_to_plot, metric_titles)):
    ax = axes[idx // 2, idx % 2]
    
    # Get data for plotting
    labels = prediction_performance_mode_df['label'].tolist()
    mode_values = prediction_performance_mode_df[metric].tolist()
    last_values = prediction_performance_last_df[metric].tolist()
    
    x = np.arange(len(labels))
    width = 0.35
    
    # Plot bars
    bars1 = ax.bar(x - width/2, mode_values, width, label='Mode', alpha=0.8, color='steelblue')
    bars2 = ax.bar(x + width/2, last_values, width, label='Last Frame', alpha=0.8, color='coral')
    
    # Add human baseline
    if 'avg_human_metrics' in locals():
        human_metric_name = metric if metric != 'f1_score' else 'f1'
        human_value = avg_human_metrics[human_metric_name]
        ax.axhline(y=human_value, color='red', linestyle='--', label='Human Avg', linewidth=2)
    
    ax.set_xlabel('Model Configuration', fontsize=11)
    ax.set_ylabel(title, fontsize=11)
    ax.set_title(f'{title} vs Ground Truth', fontsize=12, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=90, ha='right', fontsize=7)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('mode_vs_last_frame_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nVisualization saved to 'mode_vs_last_frame_comparison.png'")

In [ ]:
# ===================================================================
# Additional Analysis Functions (Optional)
# ===================================================================
print("\n" + "="*50)
print("Analysis Complete")
print("="*50)
print("All model evaluations have been completed.")
print("Results are saved in CSV files for further analysis.")

# Summary of saved files
print("\nSaved files:")
print("1. prediction_performance_mode.csv - MODE predictions vs ground truth (with prompt & framerate)")
print("2. prediction_performance_last.csv - LAST FRAME predictions vs ground truth (with prompt & framerate)") 
print("3. model_vs_human_agreement_mode.csv - MODE predictions vs individual humans (with prompt & framerate)")
print("4. model_vs_human_agreement_last.csv - LAST FRAME predictions vs individual humans (with prompt & framerate)")
print("5. mode_vs_last_comparison.csv - Direct comparison between methods (with prompt & framerate)")

# Optional: Create a final summary table
if 'prediction_performance_mode_df' in locals() and 'prediction_performance_last_df' in locals():
    print("\n" + "="*50)
    print("Final Summary Table")
    print("="*50)
    
    # Combine the key metrics
    summary_data = []
    
    for (model, prompt, framerate) in prediction_performance_mode_df.groupby(['model', 'prompt', 'framerate']).groups.keys():
        mode_row = prediction_performance_mode_df[
            (prediction_performance_mode_df['model'] == model) & 
            (prediction_performance_mode_df['prompt'] == prompt) & 
            (prediction_performance_mode_df['framerate'] == framerate)
        ].iloc[0]
        
        last_row = prediction_performance_last_df[
            (prediction_performance_last_df['model'] == model) & 
            (prediction_performance_last_df['prompt'] == prompt) & 
            (prediction_performance_last_df['framerate'] == framerate)
        ].iloc[0]
        
        summary_data.append({
            'Model': model,
            'Prompt': prompt,
            'Framerate': framerate,
            'Mode_Accuracy': mode_row['accuracy'],
            'Last_Accuracy': last_row['accuracy'],
            'Mode_F1': mode_row['f1_score'],
            'Last_F1': last_row['f1_score'],
            'Mode_Poorly_Ratio': mode_row['poorly_ratio'],
            'Last_Poorly_Ratio': last_row['poorly_ratio']
        })
    
    summary_df = pd.DataFrame(summary_data)
    print(summary_df.round(3))
    
    # Save summary
    summary_df.to_csv('final_summary.csv', index=False)
    print("\nFinal summary saved to 'final_summary.csv'")